In [1]:
import pandas as pd
import numpy as np
import os

print("Pandas analysis started successfully!")

Pandas analysis started successfully!


In [2]:
data_folder = r"P:\OneDrive\Documents\TMDB_Movie_Analytics\data"

movies = pd.read_csv(os.path.join(data_folder, "movies_cleaned.csv"))
genres = pd.read_csv(os.path.join(data_folder, "genres_cleaned.csv"))
movie_genres = pd.read_csv(os.path.join(data_folder, "movie_genres_cleaned.csv"))
cast = pd.read_csv(os.path.join(data_folder, "cast_cleaned.csv"))
crew = pd.read_csv(os.path.join(data_folder, "crew_cleaned.csv"))
movie_keywords = pd.read_csv(os.path.join(data_folder, "movie_keywords_cleaned.csv"))

print("All six cleaned datasets loaded successfully!")

All six cleaned datasets loaded successfully!


In [3]:
datasets = {
    "Movies": movies,
    "Genres": genres,
    "Movie Genres": movie_genres,
    "Cast": cast,
    "Crew": crew,
    "Movie Keywords": movie_keywords
}

for name, data in datasets.items():
    print(f"{name}: {data.shape[0]} rows and {data.shape[1]} columns")

Movies: 2503 rows and 12 columns
Genres: 19 rows and 2 columns
Movie Genres: 6878 rows and 2 columns
Cast: 24902 rows and 6 columns
Crew: 7289 rows and 6 columns
Movie Keywords: 41681 rows and 3 columns


In [4]:
print("Folder path:", data_folder)
print("Does this folder exist?", os.path.exists(data_folder))
print("Files inside the folder:", os.listdir(data_folder)[:6])

Folder path: P:\OneDrive\Documents\TMDB_Movie_Analytics\data
Does this folder exist? True
Files inside the folder: ['cast.csv', 'cast_cleaned.csv', 'crew.csv', 'crew_cleaned.csv', 'genres.csv', 'genres_cleaned.csv']


In [5]:
print("UNIQUE RECORD COUNTS")
print("----------------------------")

print("Unique Movies:", movies["movie_id"].nunique())

print("Unique Genres:", genres["genre_id"].nunique())

print("Unique Cast Members:", cast["person_id"].nunique())

print("Unique Crew Members:", crew["person_id"].nunique())

UNIQUE RECORD COUNTS
----------------------------
Unique Movies: 2503
Unique Genres: 19
Unique Cast Members: 11300
Unique Crew Members: 2848


In [6]:
print("BUDGET AND REVENUE CHECK")
print("----------------------------")

print("Missing budget values:", movies["budget"].isna().sum())
print("Missing revenue values:", movies["revenue"].isna().sum())

print("Zero budget values:", (movies["budget"] == 0).sum())
print("Zero revenue values:", (movies["revenue"] == 0).sum())

BUDGET AND REVENUE CHECK
----------------------------
Missing budget values: 0
Missing revenue values: 0
Zero budget values: 90
Zero revenue values: 102


In [7]:
# Select movies with valid budget and revenue
financial_movies = movies[
    (movies["budget"] > 0) &
    (movies["revenue"] > 0)
].copy()

# Calculate profit and return on investment
financial_movies["profit"] = (
    financial_movies["revenue"] - financial_movies["budget"]
)

financial_movies["roi"] = (
    financial_movies["profit"] / financial_movies["budget"]
) * 100

print("FINANCIAL PERFORMANCE ANALYSIS")
print("--------------------------------")
print("Movies included:", len(financial_movies))
print("Average profit:", round(financial_movies["profit"].mean(), 2))
print("Average ROI:", round(financial_movies["roi"].mean(), 2), "%")

FINANCIAL PERFORMANCE ANALYSIS
--------------------------------
Movies included: 2370
Average profit: 164891397.04
Average ROI: 1319.85 %


In [8]:
top_roi_movies = (
    financial_movies[
        financial_movies["budget"] >= 1000000
    ]
    .sort_values(by="roi", ascending=False)
    [["title", "budget", "revenue", "profit", "roi"]]
    .head(15)
)

print("TOP 15 MOVIES BY ROI")
print("----------------------------")
display(top_roi_movies)

TOP 15 MOVIES BY ROI
----------------------------


,title,budget,revenue,profit,roi
1767,Dragon Ball Super: Broly,1000000.0,125002821.0,124002821.0,12400.282100
535,Snow White and the Seven Dwarfs,1488423.0,184925486.0,183437063.0,12324.256142
1791,The Rocky Horror Picture Show,1400000.0,171181400.0,169781400.0,12127.242857
467,Rocky,1000000.0,117253345.0,116253345.0,11625.334500
1227,Gone with the Wind,4000000.0,402352579.0,398352579.0,9958.814475
716,The Jungle Book,4000000.0,378000000.0,374000000.0,9350.000000
635,Cinderella,2900000.0,263600000.0,260700000.0,8989.655172
372,Saw,1200000.0,104045735.0,102845735.0,8570.477917
719,One Hundred and One Dalmatians,3600000.0,303000000.0,299400000.0,8316.666667
263,E.T. the Extra-Terrestrial,10500000.0,797307407.0,786807407.0,7493.403876


In [9]:
# Extract release year
movies["release_year"] = pd.to_datetime(
    movies["release_date"]
).dt.year

# Count movies released each year
yearly_releases = (
    movies["release_year"]
    .value_counts()
    .sort_index()
)

print("MOVIE RELEASES BY YEAR")
print("----------------------------")
display(yearly_releases)

MOVIE RELEASES BY YEAR
----------------------------


release_year
1921     1
1922     1
1927     1
1931     2
1936     1
        ..
2022    65
2023    62
2024    43
2025    26
2026     8
Name: count, Length: 89, dtype: int64

In [10]:
print("Year with the most movie releases:")
print("Year:", yearly_releases.idxmax())
print("Number of movies:", yearly_releases.max())

Year with the most movie releases:
Year: 2016
Number of movies: 121


In [11]:
# Consider movies with a positive runtime
valid_runtime = movies[movies["runtime"] > 0].copy()

# Calculate quartiles
Q1 = valid_runtime["runtime"].quantile(0.25)
Q3 = valid_runtime["runtime"].quantile(0.75)

# Calculate IQR limits
IQR = Q3 - Q1
lower_limit = Q1 - (1.5 * IQR)
upper_limit = Q3 + (1.5 * IQR)

# Identify runtime outliers
runtime_outliers = valid_runtime[
    (valid_runtime["runtime"] < lower_limit) |
    (valid_runtime["runtime"] > upper_limit)
]

print("RUNTIME OUTLIER ANALYSIS")
print("----------------------------")
print("Lower limit:", round(lower_limit, 2), "minutes")
print("Upper limit:", round(upper_limit, 2), "minutes")
print("Number of runtime outliers:", len(runtime_outliers))

display(
    runtime_outliers[
        ["title", "runtime"]
    ].sort_values(by="runtime", ascending=False).head(10)
)

RUNTIME OUTLIER ANALYSIS
----------------------------
Lower limit: 58.5 minutes
Upper limit: 166.5 minutes
Number of runtime outliers: 60


,title,runtime
324,Zack Snyder's Justice League,242
1227,Gone with the Wind,238
829,Once Upon a Time in America,229
1610,Lawrence of Arabia,228
1774,Ben-Hur,212
587,The Irishman,209
1277,Seven Samurai,207
1304,Killers of the Flower Moon,206
187,The Godfather Part II,202
20,The Lord of the Rings: The Return of the King,201


In [12]:
# Join movies with their genre information
genre_popularity = movie_genres.merge(
    genres, on="genre_id"
).merge(
    movies, on="movie_id"
)

# Calculate average popularity for each genre
genre_popularity_summary = (
    genre_popularity
    .groupby("genre_name")["popularity"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

print("AVERAGE POPULARITY BY GENRE")
print("----------------------------")
display(genre_popularity_summary)

AVERAGE POPULARITY BY GENRE
----------------------------


,genre_name,popularity
0,Science Fiction,19.809216
1,Adventure,17.372862
2,Animation,16.463625
3,Action,15.809179
4,Fantasy,14.740167
5,Family,14.495491
6,Horror,13.360057
7,Music,13.142967
8,War,13.129806
9,Mystery,13.056377


In [13]:
# Connect cast information with movie ratings
actor_movies = cast.merge(
    movies[["movie_id", "vote_average"]],
    on="movie_id"
)

# Summarise movie count and average rating for each actor
actor_summary = (
    actor_movies
    .groupby("actor_name")
    .agg(
        movie_count=("movie_id", "nunique"),
        average_rating=("vote_average", "mean")
    )
    .sort_values(
        by="movie_count",
        ascending=False
    )
    .head(15)
    .reset_index()
)

print("TOP 15 ACTORS BY MOVIE COUNT")
print("----------------------------")
display(actor_summary)

TOP 15 ACTORS BY MOVIE COUNT
----------------------------


,actor_name,movie_count,average_rating
0,Samuel L. Jackson,39,6.938692
1,Robert De Niro,38,7.228316
2,Brad Pitt,38,7.257289
3,Johnny Depp,37,6.856973
4,Tom Hanks,33,7.289758
5,Willem Dafoe,32,7.114781
6,Mark Wahlberg,32,6.630063
7,Scarlett Johansson,32,7.056344
8,Tom Cruise,31,7.040613
9,Morgan Freeman,31,6.963742


In [14]:
# Find titles that appear more than once
duplicate_titles = movies[
    movies.duplicated(
        subset=["title"],
        keep=False
    )
].copy()

# Sort by title and release year
duplicate_titles = duplicate_titles.sort_values(
    by=["title", "release_year"]
)

print("DUPLICATE MOVIE TITLES")
print("----------------------------")
print("Number of movie records with repeated titles:",
      len(duplicate_titles))

display(
    duplicate_titles[
        ["title", "release_year", "movie_id"]
    ].head(20)
)

DUPLICATE MOVIE TITLES
----------------------------
Number of movie records with repeated titles: 100


,title,release_year,movie_id
878,A Nightmare on Elm Street,1984,377
1928,A Nightmare on Elm Street,2010,23437
261,Aladdin,1992,812
339,Aladdin,2019,420817
792,Alice in Wonderland,1951,12092
166,Alice in Wonderland,2010,12155
334,Beauty and the Beast,1991,10020
141,Beauty and the Beast,2017,321612
1249,Carrie,1976,7340
1116,Carrie,2013,133805


In [15]:
# Calculate correlation between popularity and vote count
popularity_correlation = movies[
    ["popularity", "vote_count"]
].corr()

print("POPULARITY AND VOTE COUNT ANALYSIS")
print("-----------------------------------")

print(
    "Correlation:",
    round(
        popularity_correlation.loc["popularity", "vote_count"],
        3
    )
)

POPULARITY AND VOTE COUNT ANALYSIS
-----------------------------------
Correlation: 0.361
